In [ ]:
from pathlib import Path
import json
import math
import pandas as pd
import matplotlib.pyplot as plt


# Experiment selectors. These match the names used by scripts/run_gamma_experiment.py.
basis = 'random_fourier'  # Options: 'taylor', 'random_relu', 'random_fourier'
weight_init = 'random'  # Options: 'random', 'zero'

# Optional overrides. Leave as None to use the defaults implied by basis and weight_init.
results_dir_override = None
gamma_grid_path_override = None
adaptive_results_dir_override = None

# Select gammas explicitly, by row indices, or by inclusive row range in the matching gamma-grid CSV.
# Row numbers below are zero-based, matching pandas DataFrame indices.
selected_gammas = None  # Example: [20051.5, 40103, 80206]
# expected objective: [0, 7, 9, 12] fourier, [0, 4, 6, 9] relu
selected_gamma_indices = [0, 7, 9, 12]  # fourier
# selected_gamma_indices = [0, 4, 6, 9] # relu
selected_gamma_row_start = None
selected_gamma_row_end = None
# selected_gamma_row_start = None
# selected_gamma_row_end = None

# Adaptive-gamma overlay selector. These three parameters choose one adaptive run.
include_adaptive_gamma = True
adaptive_gamma_start = 89.25 # fourier
# adaptive_gamma_start = 368.15625 # relu
# threshold(gamma) = threshold_ref * clip((gamma / gamma_ref)^alpha, scale_min, scale_max)
adaptive_threshold_ref = 7  # Confirmed against the manuscript figure; see docs/FOURIER_THRESHOLD.md.
adaptive_threshold_alpha = 0.1 
adaptive_saturation_metric = 'gradient'  # Options: 'gradient', 'objective'
adaptive_threshold_scale_min = 0.01
adaptive_threshold_scale_max = 100.0

# Optional run-control selector for display only. Leave as None to ignore.
adaptive_max_epochs_per_value = 500 # 500 if fourier, 5000 if relu

max_epoch = 500 # 500 if fourier, 10000 if relu
font_size = 55

basis_grid_stem = {
    'taylor': 'taylor',
    'random_relu': 'relu',
    'random_fourier': 'fourier',
}


def default_gamma_grid_path(basis):
    if basis not in basis_grid_stem:
        raise ValueError(f'Unsupported basis: {basis}')
    basis_name = basis_grid_stem[basis]
    basis_grid_path = Path(f'savio/gamma_grid_{basis_name}.csv')
    if basis_grid_path.exists():
        return basis_grid_path
    legacy_grid_path = Path('savio/gamma_grid.csv')
    if legacy_grid_path.exists():
        return legacy_grid_path
    raise FileNotFoundError(f'Could not find {basis_grid_path} or {legacy_grid_path}')


def default_results_dir(basis, weight_init):
    if weight_init not in {'random', 'zero'}:
        raise ValueError(f'Unsupported weight_init: {weight_init}')
    if basis == 'taylor':
        candidates = [Path(f'results/gamma_sweep_sgd_{weight_init}_init')]
        if weight_init == 'random':
            candidates.append(Path('results/gamma_sweep_sgd'))  # Legacy default before weight_init was explicit.
    elif basis in {'random_relu', 'random_fourier'}:
        candidates = [Path(f'results/gamma_sweep_sgd_{basis}_{weight_init}_init')]
    else:
        raise ValueError(f'Unsupported basis: {basis}')

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def default_adaptive_results_dir(basis, weight_init):
    if weight_init not in {'random', 'zero'}:
        raise ValueError(f'Unsupported weight_init: {weight_init}')
    if basis == 'taylor':
        return Path(f'results/adaptive_gamma_sgd_{weight_init}_init')
    if basis in {'random_relu', 'random_fourier'}:
        return Path(f'results/adaptive_gamma_sgd_{basis}_{weight_init}_init')
    raise ValueError(f'Unsupported basis: {basis}')


def gamma_key(value):
    return f'{float(value):.12g}'


def float_close(left, right, rel_tol=1e-9, abs_tol=1e-12):
    return math.isclose(float(left), float(right), rel_tol=rel_tol, abs_tol=abs_tol)


def selected_gammas_from_grid_rows(gamma_grid, row_start, row_end):
    if row_start is None and row_end is None:
        return None
    if row_start is None or row_end is None:
        raise ValueError('Set both selected_gamma_row_start and selected_gamma_row_end, or set both to None.')
    if row_start < 0 or row_end < row_start or row_end >= len(gamma_grid):
        raise ValueError(
            f'Invalid gamma-grid row range [{row_start}, {row_end}] for grid with {len(gamma_grid)} rows.'
        )
    return gamma_grid.iloc[row_start:row_end + 1]['gamma'].astype(float).tolist()


def selected_gammas_from_grid_indices(gamma_grid, row_indices):
    if row_indices is None:
        return None
    invalid_indices = [idx for idx in row_indices if idx < 0 or idx >= len(gamma_grid)]
    if invalid_indices:
        raise ValueError(
            f'Invalid gamma-grid row indices {invalid_indices} for grid with {len(gamma_grid)} rows.'
        )
    return gamma_grid.iloc[row_indices]['gamma'].astype(float).tolist()


def normalize_history_columns(history):
    # Prefer explicit objective column names, fall back to legacy aliases.
    if 'objective_train_gamma_regularized' not in history.columns and 'loss_train_reg' in history.columns:
        history['objective_train_gamma_regularized'] = history['loss_train_reg']
    if 'objective_train_unregularized' not in history.columns and 'loss_train' in history.columns:
        history['objective_train_unregularized'] = history['loss_train']
    if 'objective_val_unregularized' not in history.columns and 'loss_val' in history.columns:
        history['objective_val_unregularized'] = history['loss_val']
    return history


def read_adaptive_events(summary_path):
    events_path = summary_path.with_name('adaptive_gamma_events.csv')
    if not events_path.exists() or events_path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(events_path)


def load_adaptive_gamma_run(adaptive_results_dir):
    matches = []
    for summary_path in sorted(adaptive_results_dir.glob('adaptive_gamma_start_*/summary.json')):
        with summary_path.open() as f:
            summary = json.load(f)
        adaptive_summary = summary.get('adaptive_gamma', {})
        if not adaptive_summary.get('enabled', False):
            continue
        if summary.get('basis') != basis or summary.get('weight_init') != weight_init:
            continue
        if adaptive_saturation_metric is not None and adaptive_summary.get('saturation_metric') != adaptive_saturation_metric:
            continue
        if not float_close(adaptive_summary.get('start_gamma'), adaptive_gamma_start):
            continue
        threshold_ref = adaptive_summary.get(
            'saturation_threshold_ref',
            adaptive_summary.get('saturation_threshold'),
        )
        if not float_close(threshold_ref, adaptive_threshold_ref):
            continue
        if not float_close(adaptive_summary.get('saturation_threshold_alpha'), adaptive_threshold_alpha):
            continue
        if adaptive_threshold_scale_min is not None and not float_close(
            adaptive_summary.get('saturation_threshold_scale_min'),
            adaptive_threshold_scale_min,
        ):
            continue
        if adaptive_threshold_scale_max is not None and not float_close(
            adaptive_summary.get('saturation_threshold_scale_max'),
            adaptive_threshold_scale_max,
        ):
            continue
        if (
            adaptive_max_epochs_per_value is not None
            and adaptive_summary.get('max_epochs_per_value') != adaptive_max_epochs_per_value
        ):
            continue

        history = pd.read_csv(summary_path.with_name('history.csv'))
        history = normalize_history_columns(history)
        history = history[history['epoch_index'] <= max_epoch].copy()
        events = read_adaptive_events(summary_path)
        matches.append((summary_path, summary, history, events))

    if len(matches) > 1:
        raise ValueError(
            'More than one adaptive-gamma run matched the selected '
            'start gamma, threshold_ref, and threshold_alpha: '
            + ', '.join(str(path.parent) for path, _summary, _history, _events in matches)
        )
    if not matches:
        raise ValueError(
            'No adaptive-gamma run matched '
            f'basis={basis}, weight_init={weight_init}, start_gamma={adaptive_gamma_start}, '
            f'threshold_ref={adaptive_threshold_ref}, threshold_alpha={adaptive_threshold_alpha} '
            f'in {adaptive_results_dir}'
        )
    return matches[0][1], matches[0][2], matches[0][3]


results_dir = Path(results_dir_override) if results_dir_override is not None else default_results_dir(basis, weight_init)
adaptive_results_dir = (
    Path(adaptive_results_dir_override)
    if adaptive_results_dir_override is not None
    else default_adaptive_results_dir(basis, weight_init)
)
gamma_grid_path = (
    Path(gamma_grid_path_override)
    if gamma_grid_path_override is not None
    else default_gamma_grid_path(basis)
)

gamma_grid = pd.read_csv(gamma_grid_path)
if selected_gammas is None:
    selected_gammas = selected_gammas_from_grid_indices(gamma_grid, selected_gamma_indices)
if selected_gammas is None:
    selected_gammas = selected_gammas_from_grid_rows(
        gamma_grid,
        selected_gamma_row_start,
        selected_gamma_row_end,
    )

print(f'Basis: {basis}')
print(f'Weight initialization: {weight_init}')
print(f'Gamma grid: {gamma_grid_path}')
print(f'Results directory: {results_dir}')
print(f'Selected gammas: {selected_gammas if selected_gammas is not None else "all"}')
if include_adaptive_gamma:
    print(f'Adaptive results directory: {adaptive_results_dir}')
    print(
        'Adaptive selector: '
        f'start_gamma={adaptive_gamma_start}, '
        f'threshold_ref={adaptive_threshold_ref}, '
        f'threshold_alpha={adaptive_threshold_alpha}'
    )

grid_gamma_keys = [gamma_key(value) for value in gamma_grid['gamma']]
grid_gamma_lookup = {
    gamma_key(value): float(value)
    for value in gamma_grid['gamma']
}

selected_gamma_keys = None
if selected_gammas is not None:
    selected_gamma_keys = {gamma_key(value) for value in selected_gammas}
    unknown_gammas = sorted(selected_gamma_keys - set(grid_gamma_keys))
    if unknown_gammas:
        raise ValueError(
            f'Selected gammas are not present in {gamma_grid_path}: '
            + ', '.join(unknown_gammas)
        )

runs = []
seen_gamma_keys = set()
for summary_path in sorted(results_dir.glob('gamma_*/summary.json')):
    with summary_path.open() as f:
        summary = json.load(f)
    gamma = float(summary['gamma'])
    key = gamma_key(gamma)
    if key not in grid_gamma_lookup:
        continue
    if selected_gamma_keys is not None and key not in selected_gamma_keys:
        continue
    history = pd.read_csv(summary_path.with_name('history.csv'))
    history = normalize_history_columns(history)
    history = history[history['epoch_index'] <= max_epoch].copy()
    runs.append((grid_gamma_lookup[key], history))
    seen_gamma_keys.add(key)

runs = sorted(runs, key=lambda item: item[0])
expected_gamma_keys = selected_gamma_keys if selected_gamma_keys is not None else set(grid_gamma_keys)
missing_gamma_keys = sorted(expected_gamma_keys - seen_gamma_keys)
if missing_gamma_keys:
    raise ValueError(
        f'Missing gamma-sweep results in {results_dir}: ' + ', '.join(missing_gamma_keys)
    )
if not runs:
    raise ValueError(f'No matching gamma-sweep histories found in {results_dir}')

adaptive_run = None
if include_adaptive_gamma:
    adaptive_run = load_adaptive_gamma_run(adaptive_results_dir)




In [ ]:

plot_colors = ["#B30000", "#006600", "#0000FF", "#D100D1"]
adaptive_plot_color = '#000000'
adaptive_plot_marker = '^'
plot_xlabel = '# gradient calls'
plot_figsize = (22, 18)
plot_line_style = '-.'
adaptive_plot_line_style = '--'
plot_linewidth = 10
plot_markersize = 10
plot_alpha = 0.8

regularized_train_title = None
regularized_train_ylabel = 'Regularized Objective'
unregularized_train_title = None
unregularized_train_ylabel = 'Unregularized Training Objective'
unregularized_val_title = None
unregularized_val_ylabel = 'Unregularized Objective'


def adaptive_gamma_label(summary):
    adaptive_summary = summary['adaptive_gamma']
    return (
        'adaptive gamma'
        # fr'$\gamma_0$={adaptive_summary["start_gamma"]:g}, '
        # fr'$\epsilon_0$={adaptive_summary["saturation_threshold_ref"]:g}, '
        # fr'$\alpha$={adaptive_summary["saturation_threshold_alpha"]:g}'
    )


def plot_gamma_history(metric_col, title, ylabel):
    fig, ax = plt.subplots(figsize=plot_figsize)

    for color_index, (gamma, history) in enumerate(runs):
        ax.plot(
            history['epoch_index'],
            history[metric_col],
            plot_line_style,
            color=plot_colors[color_index % len(plot_colors)],
            linewidth=plot_linewidth,
            markersize=plot_markersize,
            # markevery=10,
            alpha=plot_alpha,
            label=fr'$\gamma$={gamma:g}',
        )

    if adaptive_run is not None:
        adaptive_summary, adaptive_history, adaptive_events = adaptive_run
        ax.plot(
            adaptive_history['epoch_index'],
            adaptive_history[metric_col],
            linestyle=adaptive_plot_line_style,
            marker=adaptive_plot_marker,
            color=adaptive_plot_color,
            linewidth=plot_linewidth,
            markersize=30,
            markevery=10,
            alpha=plot_alpha,
            label=adaptive_gamma_label(adaptive_summary),
        )
        if not adaptive_events.empty:
            for event_index, shrink_epoch in enumerate(adaptive_events['epoch_index']):
                ax.axvline(
                    shrink_epoch,
                    color=adaptive_plot_color,
                    linestyle=':',
                    linewidth=3,
                    alpha=0.6,
                    # label='adaptive gamma shrink' if event_index == 0 else None,
                )

    ax.set_title(title, fontsize=font_size)
    ax.set_xlabel(plot_xlabel, fontsize=font_size)
    ax.set_ylabel(ylabel, fontsize=font_size)
    ax.yaxis.label.set_size(font_size)
    ax.set_xlim(left=0, right=max_epoch)
    ax.set_yscale('log')
    ax.tick_params(axis='both', which='both', labelsize=font_size)
    for tick_label in ax.get_xticklabels(which='both') + ax.get_yticklabels(which='both'):
        tick_label.set_fontsize(font_size)
    ax.yaxis.get_offset_text().set_fontsize(font_size)
    ax.xaxis.get_offset_text().set_fontsize(font_size)
    ax.grid(True)
    ax.legend(fontsize=font_size)

    plt.tight_layout()
    plt.show()




In [ ]:

plot_gamma_history(
    'objective_train_gamma_regularized',
    regularized_train_title,
    regularized_train_ylabel,
)


In [ ]:

plot_gamma_history(
    'objective_train_unregularized',
    unregularized_train_title,
    unregularized_train_ylabel,
)


In [ ]:

plot_gamma_history(
    'objective_val_unregularized',
    unregularized_val_title,
    unregularized_val_ylabel,
)


## Worst-case gap across lambda

Plot the post-processed worst-case gap saved by the Savio second-stage jobs.


In [ ]:
max_epoch = 20000

In [ ]:
worst_case_gap_title = None
worst_case_gap_ylabel = r'Worst-case Gap'


def load_worst_case_gap_runs():
    gap_by_gamma_key = {}
    for summary_path in sorted(results_dir.glob('gamma_*/summary.json')):
        with summary_path.open() as f:
            summary = json.load(f)
        gamma = float(summary['gamma'])
        key = gamma_key(gamma)
        gap_path = summary_path.with_name('worst_case_gap.csv')
        if not gap_path.exists():
            continue
        gap_history = pd.read_csv(gap_path)
        gap_history = gap_history[gap_history['epoch_index'] <= max_epoch].copy()
        gap_by_gamma_key[key] = gap_history

    gap_runs = []
    missing_gap_keys = []
    for gamma, _history in runs:
        key = gamma_key(gamma)
        if key not in gap_by_gamma_key:
            missing_gap_keys.append(key)
            continue
        gap_runs.append((gamma, gap_by_gamma_key[key]))

    if missing_gap_keys:
        raise ValueError(
            f'Missing worst-case gap results in {results_dir}: ' + ', '.join(missing_gap_keys)
        )
    if not gap_runs:
        raise ValueError(f'No worst-case gap results found in {results_dir}')
    return gap_runs


def plot_worst_case_gap():
    gap_runs = load_worst_case_gap_runs()
    fig, ax = plt.subplots(figsize=plot_figsize)

    for color_index, (gamma, gap_history) in enumerate(gap_runs):
        ax.plot(
            gap_history['epoch_index'],
            gap_history['worst_case_gap'],
            plot_line_style,
            color=plot_colors[color_index % len(plot_colors)],
            linewidth=plot_linewidth,
            markersize=plot_markersize,
            alpha=plot_alpha,
            label=fr'$\gamma$={gamma:g}',
        )

    ax.set_title(worst_case_gap_title, fontsize=font_size)
    ax.set_xlabel(plot_xlabel, fontsize=font_size)
    ax.set_ylabel(worst_case_gap_ylabel, fontsize=font_size)
    ax.yaxis.label.set_size(font_size)
    ax.set_xlim(left=0, right=max_epoch)
    ax.set_yscale('log')
    ax.tick_params(axis='both', which='both', labelsize=font_size)
    for tick_label in ax.get_xticklabels(which='both') + ax.get_yticklabels(which='both'):
        tick_label.set_fontsize(font_size)
    ax.yaxis.get_offset_text().set_fontsize(font_size)
    ax.xaxis.get_offset_text().set_fontsize(font_size)
    ax.grid(True)
    ax.legend(fontsize=font_size)

    plt.tight_layout()
    plt.show()


plot_worst_case_gap()
